In [0]:
import mlflow
from databricks.feature_engineering import FeatureLookup, FeatureEngineeringClient
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report

In [0]:
import sklearn
import cloudpickle
import mlflow

print(f"scikit-learn: {sklearn.__version__}")
print(f"cloudpickle:  {cloudpickle.__version__}")
print(f"mlflow:       {mlflow.__version__}")

In [0]:


mlflow.set_experiment("/fraud-detection-demo")

raw_df = spark.table("fraud_demo.transactions.raw").toPandas()
features_df = (spark.table("fraud_demo.transactions.user_features")
               .toPandas())

training_df = raw_df.merge(features_df, on="user_id", how="left")

feature_cols = [
    "amount", "hour",
    "txn_count_7d", "avg_amount_7d",
    "max_amount_7d", "std_amount_7d",
    "unknown_merchant_count"
]

X = training_df[feature_cols]
y = training_df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

with mlflow.start_run(run_name="gbt_fraud_v2") as run:
    model = GradientBoostingClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42
    )
    model.fit(X_train, y_train)

    train_auc = roc_auc_score(
        y_train, model.predict_proba(X_train)[:,1])
    test_auc = roc_auc_score(
        y_test, model.predict_proba(X_test)[:,1])

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("train_auc", round(train_auc, 4))
    mlflow.log_metric("test_auc", round(test_auc, 4))
    mlflow.log_param("feature_cols", feature_cols)

    signature = infer_signature(
        X_train,
        model.predict_proba(X_train)
    )

    mlflow.sklearn.log_model(
        sk_model=model,
        name="fraud_model",
        registered_model_name=
            "fraud_demo.transactions.fraud_classifier",
        input_example=X_train.iloc[:3],
        signature=signature        # <-- added

    )

    print(f"Train AUC: {train_auc:.4f}")
    print(f"Test AUC:  {test_auc:.4f}")
    print(classification_report(
        y_test, model.predict(X_test),
        target_names=["legit","fraud"]))